# Shift-Scan: Results Analysis & Ablation Notebook

Reads the result files from your runs (lm-eval JSONs, training logs, calibration JSONs, size report) and produces the tables, figures and verdicts needed to judge the ablations **without overclaiming**. It does not train anything and needs no GPU.

**What it answers**

1. Is the data consistent enough to compare? (a protocol guard refuses to pool runs trained or evaluated differently)
2. Per condition: mean, spread and every seed's value, next to the teacher and chance level
3. Paired ablations by seed: HARPQ vs nearest, dyadic2 vs nearest, nearest vs floating-point decay, and so on, with exact tests, effect sizes, the **smallest p the design could ever reach**, and seeds needed
4. Long-context loss, training curves, decay-calibration error, and compression/energy arithmetic
5. A budget planner and the exact commands for the runs still missing, then a written report

**Rules built into the statistics** (so the notebook cannot flatter the results)

- Comparisons are **paired by seed**, and only inside one protocol.
- With *n* paired seeds the exact sign-flip test cannot go below p = 2/2^n (0.25 at n=3, 0.0625 at n=5, 0.031 at n=6), so the notebook says "insufficient seeds" instead of a verdict below 5 seeds.
- Holm correction is applied across the **declared** ablation pairs on the **primary metric only**; other metrics are exploratory.
- Partial rows (values copied from a printout, or PPL only) are flagged and never used for metrics they do not have.

Run order: edit the config cell, then *Run all*. Outputs go to `OUT_DIR` (CSV tables, PNG figures, `analysis_report.md`, `analysis_summary.json`).

In [ ]:
# ===== 1. CONFIG: edit this cell only =====
import os

# Where your result files live (Colab/Drive layout shown; change for Kaggle or local use).
ROOTS = {
    "drive":  "/content/drive/MyDrive/shift_scan_runs",   # Pilot A: eval_pilot_*.json, pilot_*.log, teacher_eval.json
    "pilotB": "/content/pilotB",                          # unzipped shift_scan_real_run_outputs.zip (or a run folder)
    "kaggle": "/kaggle/working/shift_scan_20m_runs",      # the 20M-token job (or its downloaded copy)
}
OUT_DIR = "analysis_out"
MOUNT_DRIVE = True                    # Colab only

TEACHER_JSON = "{drive}/teacher_eval.json"
SIZE_REPORT_JSON = None               # e.g. "{drive}/pilot_nearest_s1/size_report.json" (optional, used for verification)

# Pilot A files are found automatically by name: eval_pilot_<condition>_s<seed>.json (+ pilot_<condition>_s<seed>.log)
AUTO_PILOT_A = True
PILOT_A_PROTOCOL = "pilotA_10M"       # 10M tokens, 305 steps, seq 2048, eval dtype bf16

# Extra runs. A row whose files do not exist yet is skipped with a message (the 20M run may still be training).
EXTRA_RUNS = [
    dict(condition="harpq", seed=2, protocol="fast2M",
         eval_json="{pilotB}/student_fineweb_ppl.json", train_log="{pilotB}/qat_training.log",
         note="Pilot B: 2M tokens, 976 steps, seq 512, eval dtype fp16"),
    dict(condition="harpq", seed=0, protocol="run20M",
         eval_json="{kaggle}/eval_harpq_h128_ternary_20m_seed0.json",
         train_log="{kaggle}/harpq_h128_ternary_20m_seed0/log.jsonl",
         args_json="{kaggle}/harpq_h128_ternary_20m_seed0/args.json",
         note="20M tokens, seq 1024, micro-bs 1 x accum 32"),
]

# Values typed in by hand (partial rows: only the metrics listed, and never used for other metrics).
MANUAL_ROWS = [
    dict(condition="nearest", seed=0, protocol="pilotA_10M", partial=True, metrics=dict(student_ppl=109.49),
         note="from old logs; checkpoint lost, no downstream eval"),
    dict(condition="harpq",   seed=0, protocol="pilotA_10M", partial=True, metrics=dict(student_ppl=100.99),
         note="from old logs; checkpoint lost, no downstream eval"),
    dict(condition="dyadic2", seed=0, protocol="pilotA_10M", partial=True, metrics=dict(student_ppl=103.11),
         note="from old logs; checkpoint lost, no downstream eval"),
]
# Optional: the partial summary file made earlier for harpq / dyadic2 seed 2 (raw Kaggle files were lost).
PARTIAL_SUMMARY_JSON = "{drive}/seed2_results_summary.json"

# Decay-calibration JSONs (label, path)
CALIBRATIONS = [("Pilot B calibration (200k tokens)", "{pilotB}/fineweb_harpq_calibration.json")]

# Ablations to test, as (A, B): "is A better than B?". Declared up front so Holm correction is honest.
ABLATIONS = [("harpq", "nearest"), ("dyadic2", "nearest"), ("nearest", "fp_decay"),
             ("harpq", "fp_decay"), ("dyadic2", "harpq")]
PRIMARY_METRIC = "wikitext_word_ppl"
MIN_SEEDS_FOR_VERDICT = 5
ALPHA = 0.05
RNG_SEED = 0

# Architecture used for the size and energy arithmetic (Mamba-2 130M; verified against the size report below)
ARCH = dict(d_model=768, n_layer=24, d_state=128, headdim=64, expand=2, ngroups=1, vocab=50288, conv_width=4)

# Budget planner
PLAN = dict(conditions=["nearest", "harpq", "dyadic2", "fp_decay"], seeds=[0, 1, 2], tokens=19_988_480)
DEFAULT_TOK_PER_S = 920.0             # used only if no run log with tok_s is found
# The 20M run's settings, copied from your notebook's printed command
RUN20M_ARGS = dict(seq_len=1024, micro_bs=1, accum=32, lr=1e-4, warmup=50, min_lr_frac=0.1, dtype="fp16",
                   val_seqs=32, eval_every=500, save_every=100, group_size=128, state_weight=0.05)
VARIANT_FLAGS = {
    "nearest":  ["--decay", "shift", "--decay-quantizer", "nearest"],
    "harpq":    ["--decay", "shift", "--decay-quantizer", "horizon", "--decay-horizon", "128", "--decay-lag-weight", "uniform"],
    "dyadic2":  ["--decay", "shift", "--decay-quantizer", "dyadic", "--dyadic-terms", "2", "--decay-horizon", "128"],
    "fp_decay": ["--decay", "exp"],
}


In [ ]:
# ===== 2. Imports and loaders =====
import os, re, io, json, ast, glob, math, time, shutil, pathlib, itertools, warnings
import numpy as np, pandas as pd
import matplotlib
try:
    from IPython import get_ipython
    IN_NB = get_ipython() is not None
except Exception:
    IN_NB = False
if not IN_NB: matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from google.colab import drive, files
    IN_COLAB = True
    if MOUNT_DRIVE: drive.mount("/content/drive")
except ImportError:
    IN_COLAB = False

OUT = pathlib.Path(OUT_DIR); (OUT / "figs").mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(RNG_SEED)

def P(path):
    return path.format(**ROOTS) if isinstance(path, str) else path
def exists(path):
    return bool(path) and os.path.exists(P(path))
def load_json(path):
    with open(P(path), "r", encoding="utf-8", errors="ignore") as f: return json.load(f)
def read_text(path):
    with open(P(path), "r", encoding="utf-8", errors="ignore") as f: return f.read()

# metric name -> (lm-eval task, key, higher_is_better)
TASK_KEYS = {
    "wikitext_word_ppl": ("wikitext", "word_perplexity,none", False),
    "wikitext_bpb":      ("wikitext", "bits_per_byte,none", False),
    "lambada_acc":       ("lambada_openai", "acc,none", True),
    "lambada_ppl":       ("lambada_openai", "perplexity,none", False),
    "hellaswag":         ("hellaswag", "acc_norm,none", True),
    "piqa":              ("piqa", "acc_norm,none", True),
    "arc_easy":          ("arc_easy", "acc_norm,none", True),
    "arc_challenge":     ("arc_challenge", "acc_norm,none", True),
    "winogrande":        ("winogrande", "acc,none", True),
}
CHANCE = {"hellaswag": 0.25, "piqa": 0.50, "arc_easy": 0.25, "arc_challenge": 0.25, "winogrande": 0.50}
HIGHER = {k: v[2] for k, v in TASK_KEYS.items()}
HIGHER.update(student_ppl=False, s_ce=False, state_mse=False)
LOG_METRICS = ["student_ppl", "s_ce", "state_mse"]
ALL_METRICS = list(TASK_KEYS) + LOG_METRICS

def extract_eval(d):
    res, out = d.get("results", {}), {}
    for name, (task, key, _) in TASK_KEYS.items():
        v = res.get(task, {}).get(key)
        if isinstance(v, (int, float)): out[name] = float(v)
        se = res.get(task, {}).get(key.replace(",none", "_stderr,none"))
        if isinstance(se, (int, float)): out[name + "_se"] = float(se)
    proto = d.get("evaluation_protocol", {})
    meta = dict(eval_dtype=str(proto.get("dtype", "?")).replace("torch.", ""), eval_limit=proto.get("limit"),
                synthetic=proto.get("synthetic_evaluation_data"), val_rule=proto.get("long_context_validation_rule"))
    return out, meta, d.get("loss_by_position", {})

def parse_text_log(path):
    train, evals = [], []
    for ln in read_text(path).splitlines():
        ln = ln.strip()
        try:
            if ln.startswith("{'step'"):
                train.append(ast.literal_eval(ln))
            elif ln.startswith("EVAL "):
                evals.append(ast.literal_eval(ln[5:]))
            else:
                m = re.match(r"^step 0 .*?: (\{.*\})$", ln)
                if m: evals.append(dict(ast.literal_eval(m.group(1)), step=0))
        except Exception:
            pass
    return train, evals

def parse_jsonl(path):
    train, evals = [], []
    for ln in read_text(path).splitlines():
        ln = ln.strip()
        if not ln: continue
        try: r = json.loads(ln)
        except Exception: continue
        if "student_ppl" in r: evals.append(r)
        elif "step" in r and "loss" in r: train.append(r)
    return train, evals

def load_logs(path):
    if not exists(path): return [], []
    return parse_jsonl(path) if str(path).endswith(".jsonl") else parse_text_log(path)

def log_summary(train, evals):
    out = {}
    if evals:
        last = evals[-1]
        for k in LOG_METRICS:
            if isinstance(last.get(k), (int, float)): out[k] = float(last[k])
    toks = [r["tok_s"] for r in train if isinstance(r.get("tok_s"), (int, float))]
    if toks: out["tok_s_median"] = float(np.median(toks))
    return out

def discover_pilot_a():
    rows, root = [], P("{drive}")
    for f in sorted(glob.glob(os.path.join(root, "eval_pilot_*_s*.json"))):
        m = re.match(r"eval_pilot_([A-Za-z0-9]+)_s(\d+)\.json$", os.path.basename(f))
        if not m: continue
        c, s = m.group(1), int(m.group(2))
        log = os.path.join(root, f"pilot_{c}_s{s}.log")
        rows.append(dict(condition=c, seed=s, protocol=PILOT_A_PROTOCOL, eval_json=f,
                         train_log=log if os.path.exists(log) else None, note="Pilot A (auto-discovered)"))
    return rows


In [ ]:
# ===== 3. Build the master table and check it =====
registry = (discover_pilot_a() if AUTO_PILOT_A else []) + list(EXTRA_RUNS)
rows, notes_skipped, problems = [], [], []

for r in registry:
    ev = r.get("eval_json")
    if not exists(ev):
        notes_skipped.append(f"skipped {r['condition']} seed {r['seed']} [{r['protocol']}]: eval file not found ({ev})"); continue
    metrics, meta, lbp = extract_eval(load_json(ev))
    train, evals = load_logs(r.get("train_log"))
    row = dict(condition=r["condition"], seed=int(r["seed"]), protocol=r["protocol"], partial=False,
               source=os.path.basename(P(ev)), note=r.get("note", ""), **meta, **metrics, **log_summary(train, evals))
    row["_lbp"] = lbp; row["_train"] = train; row["_evals"] = evals; row["_log"] = r.get("train_log")
    if exists(r.get("args_json")): row["_args"] = load_json(r["args_json"])
    rows.append(row)

for m in MANUAL_ROWS:
    rows.append(dict(condition=m["condition"], seed=int(m["seed"]), protocol=m["protocol"], partial=True,
                     source="manual", note=m.get("note", ""), **m["metrics"]))

if exists(PARTIAL_SUMMARY_JSON):
    S = load_json(PARTIAL_SUMMARY_JSON)
    for name, run in S.get("runs", {}).items():
        if not str(run.get("data_source", "")).upper().startswith("PARTIAL"): continue
        te, dn = run.get("train_eval_step_305", {}), run.get("downstream", {})
        met = {}
        if "student_ppl" in te: met["student_ppl"] = te["student_ppl"]
        if "s_ce" in te: met["s_ce"] = te["s_ce"]
        if "state_mse" in te: met["state_mse"] = te["state_mse"]
        if "acc" in dn.get("lambada_openai", {}): met["lambada_acc"] = dn["lambada_openai"]["acc"]
        if "word_perplexity" in dn.get("wikitext", {}): met["wikitext_word_ppl"] = dn["wikitext"]["word_perplexity"]
        rows.append(dict(condition=run["variant"], seed=int(S.get("seed", 2)), protocol=PILOT_A_PROTOCOL, partial=True,
                         source=os.path.basename(P(PARTIAL_SUMMARY_JSON)), note="copied from a printed summary (raw files lost)", **met))

RUNS = pd.DataFrame(rows)
if RUNS.empty:
    raise SystemExit("No runs found. Check ROOTS and the file names in the config cell.")
for c in ALL_METRICS:
    if c not in RUNS: RUNS[c] = np.nan

# duplicates: the same (protocol, condition, seed) from a full and a partial source -> keep the full one
RUNS = RUNS.sort_values("partial").drop_duplicates(["protocol", "condition", "seed"], keep="first").reset_index(drop=True)

# Teacher
TEACHER, TEACHER_META, TEACHER_LBP = {}, {}, {}
if exists(TEACHER_JSON):
    TEACHER, TEACHER_META, TEACHER_LBP = extract_eval(load_json(TEACHER_JSON))
else:
    problems.append("teacher results not found: no student number can be judged without a teacher row")

# Protocol guard
full = RUNS[~RUNS.partial]
for proto, g in full.groupby("protocol"):
    if g.eval_dtype.nunique() > 1: problems.append(f"[{proto}] runs were evaluated with different dtypes: {sorted(g.eval_dtype.unique())}")
    if g.eval_limit.notna().any(): problems.append(f"[{proto}] some evals used --limit (not full task sets)")
    if (g.synthetic == True).any(): problems.append(f"[{proto}] an eval used SYNTHETIC data")
    if TEACHER_META and TEACHER_META.get("eval_dtype") not in (None, "?") and set(g.eval_dtype) - {TEACHER_META["eval_dtype"]}:
        problems.append(f"[{proto}] eval dtype {sorted(set(g.eval_dtype))} differs from the teacher's {TEACHER_META['eval_dtype']}; gaps to the teacher are slightly inflated or deflated")

print("Runs loaded:", len(RUNS), "(", int(RUNS.partial.sum()), "partial )")
inv = RUNS.groupby(["protocol", "condition"]).agg(n=("seed", "count"), seeds=("seed", lambda s: sorted(s)),
                                                    partial=("partial", "sum")).reset_index()
print(inv.to_string(index=False))
if notes_skipped: print("\nSkipped:\n  " + "\n  ".join(notes_skipped))
if problems: print("\nPROBLEMS TO READ BEFORE TRUSTING ANYTHING:\n  - " + "\n  - ".join(problems))
else: print("\nNo protocol problems detected.")
print("\nReminder: results from different protocols are never pooled below.")


In [ ]:
# ===== 4. Per-condition results (each seed shown, never only the mean) =====
def md_table(df, floatfmt="{:.4g}"):
    cols = list(df.columns)
    out = ["| " + " | ".join(map(str, cols)) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for _, r in df.iterrows():
        out.append("| " + " | ".join(floatfmt.format(v) if isinstance(v, (float, np.floating)) and not pd.isna(v)
                                   else ("" if (isinstance(v, float) and pd.isna(v)) else str(v)) for v in r) + " |")
    return "\n".join(out)

MAIN_METRICS = [m for m in ["wikitext_word_ppl", "wikitext_bpb", "lambada_acc", "lambada_ppl", "student_ppl",
                            "hellaswag", "piqa", "arc_easy", "arc_challenge", "winogrande"]
                if RUNS[m].notna().any()]
main_rows = []
for (proto, cond), g in RUNS.groupby(["protocol", "condition"]):
    row = {"protocol": proto, "condition": cond}
    for m in MAIN_METRICS:
        v = g.dropna(subset=[m]).sort_values("seed")
        if v.empty: row[m] = ""; continue
        vals = v[m].values
        sd = f" +/- {vals.std(ddof=1):.3g}" if len(vals) > 1 else ""
        row[m] = f"{vals.mean():.4g}{sd}  (n={len(vals)}: " + ", ".join(f"s{int(s)}={x:.4g}" for s, x in zip(v.seed, vals)) + ")"
    main_rows.append(row)
MAIN = pd.DataFrame(main_rows)
MAIN.to_csv(OUT / "main_results.csv", index=False)
pd.set_option("display.max_colwidth", 200, "display.width", 250)

for proto, g in MAIN.groupby("protocol"):
    print(f"\n=== protocol: {proto} ===")
    print(g.drop(columns="protocol").to_string(index=False))

if TEACHER:
    print("\n=== teacher ===")
    print({k: round(v, 4) for k, v in TEACHER.items() if not k.endswith("_se")})

# gap to teacher and chance check, per condition (mean over full-eval seeds)
gap_rows = []
for (proto, cond), g in RUNS[~RUNS.partial].groupby(["protocol", "condition"]):
    for m in TASK_KEYS:
        if m not in TEACHER or g[m].dropna().empty: continue
        s = g[m].dropna().mean(); t = TEACHER[m]
        row = dict(protocol=proto, condition=cond, metric=m, teacher=t, student=s, ratio=s / t if t else np.nan)
        if m in CHANCE:
            se = g[m + "_se"].dropna().mean() if (m + "_se") in g else np.nan
            row.update(chance=CHANCE[m], above_chance_2se=bool(s - CHANCE[m] > 2 * se) if not np.isnan(se) else None)
        gap_rows.append(row)
GAP = pd.DataFrame(gap_rows)
if not GAP.empty:
    GAP.to_csv(OUT / "gap_to_teacher.csv", index=False)
    print("\n=== gap to teacher ===")
    print(GAP.round(4).to_string(index=False))

# chance check does not need a teacher: is each multiple-choice score above chance by more than 2 standard errors?
chance_rows = []
for (proto, cond), g in RUNS[~RUNS.partial].groupby(["protocol", "condition"]):
    for m, ch in CHANCE.items():
        if m not in g or g[m].dropna().empty: continue
        s_ = g[m].dropna().mean(); se = g[m + "_se"].dropna().mean() if (m + "_se") in g else np.nan
        chance_rows.append(dict(protocol=proto, condition=cond, task=m, student=s_, chance=ch, se=se,
                                margin_in_se=(s_ - ch) / se if se and not np.isnan(se) else np.nan,
                                above_chance_2se=bool(s_ - ch > 2 * se) if not np.isnan(se) else None, teacher=TEACHER.get(m)))
CH = pd.DataFrame(chance_rows)
if not CH.empty:
    CH.to_csv(OUT / "chance_check.csv", index=False)
    print("\n=== multiple-choice scores against chance ===")
    print(CH.round(4).to_string(index=False))
    notabove = CH[CH.above_chance_2se == False]
    if len(notabove): print("\nNOT clearly above chance:", ", ".join(f"{r.task} [{r.protocol}/{r.condition}]" for r in notabove.itertuples()))


In [ ]:
# ===== 5. Paired ablations by seed =====
def sign_flip_p(od, n_mc=200_000):
    n = len(od); obs = abs(od.mean())
    if n <= 16:
        signs = np.array(list(itertools.product([-1, 1], repeat=n)))
        means = (signs * od).mean(1)
        return float((np.abs(means) >= obs - 1e-12).mean()), 2.0 / 2 ** n
    signs = rng.choice([-1, 1], size=(n_mc, n))
    return float(((np.abs((signs * od).mean(1)) >= obs - 1e-12).sum() + 1) / (n_mc + 1)), 2.0 / 2 ** 16

def holm(p):
    p = np.asarray(p, float); m = len(p); order = np.argsort(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, p[i] * (m - rank))); adj[i] = run
    return adj

def paired_test(a, b, higher_better):
    d = np.asarray(a, float) - np.asarray(b, float); n = len(d)
    od = d if higher_better else -d                  # positive = A is better
    mean = od.mean(); sd = od.std(ddof=1) if n > 1 else np.nan
    p, pmin = sign_flip_p(od) if n >= 2 else (np.nan, np.nan)
    if n >= 3 and sd > 0:
        h = stats.t.ppf(0.975, n - 1) * sd / math.sqrt(n); ci = (mean - h, mean + h)
        need = max(3, math.ceil(((stats.norm.ppf(1 - ALPHA / 2) + stats.norm.ppf(0.8)) * sd / abs(mean)) ** 2)) if mean != 0 else np.inf
    else:
        ci, need = (od.min(), od.max()) if n else (np.nan, np.nan), np.nan
    return dict(n=n, mean_adv=mean, sd=sd, dz=mean / sd if sd and sd > 0 else np.nan, p=p, p_min=pmin,
                ci_lo=ci[0], ci_hi=ci[1], ci_is_t=(n >= 3 and sd > 0), a_better=int((od > 0).sum()), seeds_needed=need)

paired_rows = []
for proto, g in RUNS.groupby("protocol"):
    for A, B in ABLATIONS:
        for m in ALL_METRICS:
            ga = g[(g.condition == A)].dropna(subset=[m]).set_index("seed")[m]
            gb = g[(g.condition == B)].dropna(subset=[m]).set_index("seed")[m]
            common = sorted(set(ga.index) & set(gb.index))
            if len(common) < 2: continue
            t = paired_test(ga.loc[common].values, gb.loc[common].values, HIGHER[m])
            ref = gb.loc[common].mean()
            paired_rows.append(dict(protocol=proto, A=A, B=B, metric=m, seeds=common, rel_pct=100 * t["mean_adv"] / abs(ref) if ref else np.nan, **t))
PAIRED = pd.DataFrame(paired_rows)

if PAIRED.empty:
    print("No pair has at least 2 common seeds yet.")
else:
    PAIRED["holm_p"] = np.nan
    for proto, g in PAIRED[PAIRED.metric == PRIMARY_METRIC].groupby("protocol"):
        PAIRED.loc[g.index, "holm_p"] = holm(g.p.values)
    def verdict(r):
        if r.n < MIN_SEEDS_FOR_VERDICT:
            return f"insufficient seeds (n={r.n}, smallest possible p={r.p_min:.3g}): report the values, no verdict"
        if r.metric != PRIMARY_METRIC:
            return "exploratory (not the primary metric)"
        if r.holm_p < ALPHA:
            return f"{r.A} better than {r.B}" if r.mean_adv > 0 else f"{r.A} WORSE than {r.B}"
        return f"not distinguishable from seed noise (needs about {r.seeds_needed} seeds at this effect)"
    PAIRED["verdict"] = PAIRED.apply(verdict, axis=1)
    PAIRED.to_csv(OUT / "paired_ablations.csv", index=False)
    show = PAIRED[["protocol", "A", "B", "metric", "n", "mean_adv", "rel_pct", "dz", "a_better", "p", "p_min", "holm_p", "seeds_needed", "verdict"]]
    print("mean_adv > 0 means A is better (differences are oriented so that lower-is-better metrics flip sign).")
    print("a_better = number of seeds in which A beat B.\n")
    for proto, g in show.groupby("protocol"):
        print(f"=== protocol: {proto} ===")
        prim = g[g.metric.isin([PRIMARY_METRIC, "student_ppl"])]
        print(prim.round(4).to_string(index=False))


In [ ]:
# ===== 6. Figures =====
def savefig(name):
    plt.tight_layout(); plt.savefig(OUT / "figs" / name, dpi=140)
    if IN_NB: plt.show()
    plt.close()

COLORS = {"nearest": "#4C72B0", "harpq": "#DD8452", "dyadic2": "#55A868", "fp_decay": "#8172B3"}

# (a) every seed per condition, per protocol
for metric in [PRIMARY_METRIC, "student_ppl"]:
    for proto, g in RUNS.groupby("protocol"):
        gm = g.dropna(subset=[metric])
        if gm.empty or gm.condition.nunique() < 1: continue
        ORDER = ["fp_decay", "nearest", "harpq", "dyadic2"]
        conds = sorted(gm.condition.unique(), key=lambda c: (ORDER.index(c) if c in ORDER else 99, c)); fig, ax = plt.subplots(figsize=(1.6 * len(conds) + 3, 4))
        for i, c in enumerate(conds):
            v = gm[gm.condition == c].sort_values("seed")
            xs = np.full(len(v), i) + rng.uniform(-.1, .1, len(v))
            ax.scatter(xs, v[metric], s=55, c=[COLORS.get(c, "gray")], edgecolor=["k" if p else "none" for p in v.partial], zorder=3)
            for x_, (_, r) in zip(xs, v.iterrows()): ax.annotate(f"s{int(r.seed)}", (x_, r[metric]), xytext=(7, 0), textcoords="offset points", fontsize=7)
            ax.hlines(v[metric].mean(), i - .25, i + .25, color="k", lw=2)
        if metric in TEACHER: ax.axhline(TEACHER[metric], color="gray", ls="--", lw=1, label="teacher")
        ax.set_xticks(range(len(conds))); ax.set_xticklabels(conds); ax.set_ylabel(metric); ax.set_title(f"{metric}, protocol {proto}\n(each dot is one seed; bar = mean; black edge = partial row)", fontsize=9)
        if metric in TEACHER: ax.legend(fontsize=7)
        savefig(f"seeds_{metric}_{proto}.png")

# (b) paired differences with the smallest-possible-p reminder
if not PAIRED.empty:
    for proto, g in PAIRED[PAIRED.metric == PRIMARY_METRIC].groupby("protocol"):
        fig, ax = plt.subplots(figsize=(7, 0.7 * len(g) + 1.5))
        for i, (_, r) in enumerate(g.iterrows()):
            ax.errorbar(r.mean_adv, i, xerr=[[r.mean_adv - r.ci_lo], [r.ci_hi - r.mean_adv]] if r.ci_is_t else None, fmt="o", color="k", capsize=3)
            ax.annotate(f"n={r.n}, A better in {r.a_better}/{r.n} seeds", (r.mean_adv, i), xytext=(6, 6), textcoords="offset points", fontsize=7)
        ax.axvline(0, color="gray", ls="--"); ax.set_yticks(range(len(g))); ax.set_yticklabels([f"{a} vs {b}" for a, b in zip(g.A, g.B)])
        ax.set_xlabel(f"advantage of A on {PRIMARY_METRIC} (positive = A better); bars are 95% t-intervals when n >= 3")
        ax.set_title(f"Paired ablations, protocol {proto}", fontsize=9); savefig(f"paired_{proto}.png")

# (c) loss by position (long-context behaviour)
lbp_runs = [(f"{r.condition} s{int(r.seed)} [{r.protocol}]", r._lbp) for _, r in RUNS.iterrows() if isinstance(r.get("_lbp"), dict) and r["_lbp"]]
if lbp_runs or TEACHER_LBP:
    fig, ax = plt.subplots(figsize=(7, 4))
    if TEACHER_LBP: ax.plot(list(TEACHER_LBP), list(TEACHER_LBP.values()), "k--o", label="teacher")
    for lab, d in lbp_runs: ax.plot(list(d), list(d.values()), "-o", label=lab, alpha=.8)
    ax.set_xlabel("token position bucket"); ax.set_ylabel("loss"); ax.legend(fontsize=6); ax.set_title("Loss by position (flat = no degradation with length)", fontsize=9)
    savefig("loss_by_position.png")
    deg = [dict(run=lab, first=list(d.values())[0], last=list(d.values())[-1], last_over_first=list(d.values())[-1] / list(d.values())[0]) for lab, d in lbp_runs]
    if TEACHER_LBP: deg.insert(0, dict(run="teacher", first=list(TEACHER_LBP.values())[0], last=list(TEACHER_LBP.values())[-1], last_over_first=list(TEACHER_LBP.values())[-1] / list(TEACHER_LBP.values())[0]))
    print(pd.DataFrame(deg).round(3).to_string(index=False))

# (d) training curves
curved = [r for _, r in RUNS.iterrows() if isinstance(r.get("_train"), list) and len(r["_train"]) > 1]
if curved:
    fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))
    for r in curved:
        lab = f"{r.condition} s{int(r.seed)} [{r.protocol}]"; tr = r["_train"]
        x = [t.get("tokens", t["step"]) for t in tr]
        axs[0].plot(x, [t["loss"] for t in tr], label=lab)
        if all("state_loss" in t for t in tr): axs[1].plot(x, [t["state_loss"] for t in tr], label=lab)
        ev = r["_evals"]
        if ev: axs[2].plot([e["step"] for e in ev], [e["student_ppl"] for e in ev], "-o", label=lab)
    axs[0].set_title("train loss"); axs[1].set_title("state loss"); axs[2].set_title("held-out student PPL (see its token count!)")
    axs[0].set_yscale("log"); axs[0].set_xlabel("tokens"); axs[2].set_xlabel("step"); axs[0].legend(fontsize=6); savefig("training_curves.png")
else:
    print("No training logs with step rows found for curves.")


## 7. Decay calibration: does lower retention error explain better models?

Retention error is a property of the decay coefficients, not of model quality. The question here is only whether the ordering of conditions by retention error matches the ordering by quality. With three conditions that can never be shown statistically, so it is an exploratory view.

In [ ]:
# ===== 7. Calibration summary =====
def find_key(d, sub):
    if isinstance(d, dict):
        for k, v in d.items():
            if sub in k.lower() and isinstance(v, dict) and "retention_rmse" in v: return v
        for v in d.values():
            r = find_key(v, sub)
            if r: return r
    return None

cal_rows = []
for label, path in CALIBRATIONS:
    if not exists(path): print("calibration file not found:", path); continue
    c = load_json(path)
    n = c.get("nearest_one_step", {}) or find_key(c, "nearest") or {}
    h = c.get("horizon_aware", {}) or find_key(c, "horizon") or {}
    dy = find_key(c, "dyadic") or {}
    cal_rows.append(dict(source=label, tokens_seen=c.get("source", {}).get("tokens_seen"), coefficients=c.get("coefficients"),
                         nearest_rmse=n.get("retention_rmse"), harpq_rmse=h.get("retention_rmse"), dyadic_rmse=dy.get("retention_rmse"),
                         exponents_changed=c.get("fraction_exponents_changed_vs_nearest")))
CAL = pd.DataFrame(cal_rows)
for _c in ["tokens_seen", "coefficients", "nearest_rmse", "harpq_rmse", "dyadic_rmse", "exponents_changed"]:
    if _c in CAL: CAL[_c] = pd.to_numeric(CAL[_c], errors="coerce")
if not CAL.empty:
    CAL["harpq_vs_nearest_pct"] = 100 * (1 - CAL.harpq_rmse / CAL.nearest_rmse)
    CAL["dyadic_vs_nearest_pct"] = 100 * (1 - CAL.dyadic_rmse / CAL.nearest_rmse)
    CAL.to_csv(OUT / "calibration_summary.csv", index=False); print(CAL.round(5).to_string(index=False))

# Horizon sensitivity, copied from the research_suite S1 output you ran on the initialisation prior
# (replace with your own suite_results.json numbers if you rerun it).
HORIZON_REF = dict(horizon=[16, 32, 64, 128, 256], nearest=[0.0336] * 5, harpq=[0.0348, 0.0344, 0.0339, 0.03348, 0.03337])
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(HORIZON_REF["horizon"], HORIZON_REF["nearest"], "o-", label="nearest", color=COLORS["nearest"])
ax.plot(HORIZON_REF["horizon"], HORIZON_REF["harpq"], "o-", label="HARPQ", color=COLORS["harpq"])
ax.set_xscale("log", base=2); ax.set_xlabel("HARPQ horizon"); ax.set_ylabel("retention RMSE")
ax.set_title("HARPQ is worse than nearest at short horizons and only slightly better from 128\n(reference values from research_suite S1, initialisation prior)", fontsize=8); ax.legend()
savefig("horizon_sensitivity.png")

# retention error vs quality across conditions (exploratory)
if not CAL.empty and CAL.dyadic_rmse.notna().any():
    rmse = {"nearest": CAL.nearest_rmse.iloc[0], "harpq": CAL.harpq_rmse.iloc[0], "dyadic2": CAL.dyadic_rmse.iloc[0]}
    full_ = RUNS[(~RUNS.partial) & RUNS[PRIMARY_METRIC].notna()]
    if not full_.empty:
        proto = full_.protocol.value_counts().idxmax(); q = full_[full_.protocol == proto].groupby("condition")[PRIMARY_METRIC].mean()
        pts = [(rmse[c], q[c], c) for c in rmse if c in q.index]
        if len(pts) >= 2:
            fig, ax = plt.subplots(figsize=(5, 3.6))
            for x, y, c in pts: ax.scatter(x, y, s=80, c=COLORS.get(c, "gray")); ax.annotate(c, (x, y), xytext=(6, 4), textcoords="offset points")
            ax.set_xlabel("retention RMSE (calibration)"); ax.set_ylabel(PRIMARY_METRIC + f" [{proto}]")
            ax.set_title("Exploratory: retention error vs quality (3 points, no inference)", fontsize=8); savefig("retention_vs_quality.png")
print("\nNote: if dyadic has much lower retention error than HARPQ but does not train better, retention error is not what limits quality at this budget.")


## 8. Size and compression, with the arithmetic checked

The parameter counts below are rebuilt from the Mamba-2 130M shapes and compared with the project's own size report. If they match exactly, the byte breakdown is trustworthy rather than estimated.

In [ ]:
# ===== 8. Compression arithmetic =====
A_ = ARCH
d_inner = A_["expand"] * A_["d_model"]; nheads = d_inner // A_["headdim"]
in_out = 2 * d_inner + 2 * A_["ngroups"] * A_["d_state"] + nheads
tern = A_["n_layer"] * (A_["d_model"] * in_out + d_inner * A_["d_model"])
conv_ch = d_inner + 2 * A_["ngroups"] * A_["d_state"]
others = A_["n_layer"] * (conv_ch * A_["conv_width"] + conv_ch + 3 * nheads + A_["d_model"] + d_inner) + A_["d_model"]
emb = A_["vocab"] * A_["d_model"]
total = tern + 2 * emb + others                           # untied output head
MiB = 1024 ** 2

def model_bytes(bits_tern=2.0, group=128, scale_bits=16, emb_bits=16, head_bits=16, tie=False, others_bits=16):
    sc = lambda b: scale_bits / group if b < 16 else 0.0
    parts = dict(ternary=tern * bits_tern / 8 + tern / group * scale_bits / 8,
                 embedding=emb * (emb_bits + sc(emb_bits)) / 8,
                 head=0.0 if tie else emb * (head_bits + sc(head_bits)) / 8,
                 other=others * others_bits / 8)
    return parts, total - (emb if tie else 0)

ASIS, n_as = model_bytes()
print(f"rebuilt: total {total:,}  ternary {tern:,}  embedding+head {2 * emb:,} ({2 * emb / total:.1%} of parameters)  other {others:,}")
print(f"as-is packed size {sum(ASIS.values()) / MiB:.2f} MiB, {sum(ASIS.values()) * 8 / total:.2f} effective bits/param, "
      f"{total * 2 / sum(ASIS.values()):.2f}x vs FP16")
if SIZE_REPORT_JSON and exists(SIZE_REPORT_JSON):
    sr = load_json(SIZE_REPORT_JSON)
    ok = (sr["total_params"] == total and sr["ternary_params"] == tern and abs(sr["packed_MB"] - sum(ASIS.values()) / MiB) < 0.01)
    print("size report check:", "MATCHES EXACTLY" if ok else f"MISMATCH: report {sr}")
else:
    print("(no size report given; for the Pilot runs the report was: total 167,610,816, ternary 90,095,616, packed 170.67 MiB, 8.54 bits)")

scen = [("as-is (2-bit ternary, FP16 embedding and head)", {}),
        ("embedding and head int8", dict(emb_bits=8, head_bits=8)),
        ("embedding and head int4", dict(emb_bits=4, head_bits=4)),
        ("tied head (as the original), FP16", dict(tie=True)),
        ("tied head, int8", dict(emb_bits=8, tie=True)),
        ("tied head, int4", dict(emb_bits=4, tie=True))]
SC = []
for name, kw in scen:
    parts, npar = model_bytes(**kw); b = sum(parts.values())
    SC.append(dict(scenario=name, MiB=round(b / MiB, 2), bits_per_param=round(b * 8 / npar, 2), vs_fp16=round(total * 2 / b, 2),
                   embedding_and_head_share=f"{(parts['embedding'] + parts['head']) / b:.0%}"))
SC = pd.DataFrame(SC); SC.to_csv(OUT / "compression_scenarios.csv", index=False); print("\n" + SC.to_string(index=False))
print("\nQuantizing the embedding and head needs its own quality measurement before it is claimed; these rows are size arithmetic only.")

fig, ax = plt.subplots(figsize=(6.5, 3.4)); left = 0
for k, col in zip(["ternary", "embedding", "head", "other"], ["#55A868", "#C44E52", "#DD8452", "#999999"]):
    ax.barh(0, ASIS[k] / MiB, left=left, color=col, label=f"{k} {ASIS[k] / MiB:.1f} MiB"); left += ASIS[k] / MiB
ax.set_yticks([]); ax.set_xlabel("MiB"); ax.legend(fontsize=7); ax.set_title("Where the bytes are (as-is student)", fontsize=9); savefig("bytes_breakdown.png")


## 9. Arithmetic-energy estimate (analytic, not a measurement)

Counts the multiplies that shift-only decay removes per generated token and prices them with the widely cited 45 nm per-operation energies (Horowitz, ISSCC 2014). A shift is priced like an integer add, which is an assumption. Memory movement, which usually costs more than arithmetic, is **not** included, and nothing here replaces a hardware measurement.

In [ ]:
# ===== 9. Op counts and energy =====
E = dict(int8_add=0.03, int32_add=0.1, fp16_add=0.4, fp16_mul=1.1, int32_mul=3.1)       # picojoules
shift_cost = E["int32_add"]                                                              # assumption
state_elems = nheads * A_["headdim"] * A_["d_state"] * A_["n_layer"]                    # state elements touched per token
lin_macs = tern                                                                          # in_proj + out_proj weights = MACs per token
head_macs = emb

def uJ(pj): return pj * 1e-6
fp16_mac = E["fp16_mul"] + E["fp16_add"]
scan_fp = state_elems * (E["fp16_mul"] + E["fp16_mul"] + E["fp16_add"] + E["fp16_mul"] + E["fp16_add"])   # decay, Bx, accumulate, readout mul+add
cases = {
    "FP16 baseline (everything FP16)":
        lin_macs * fp16_mac + head_macs * fp16_mac + scan_fp,
    "ternary linear layers (8-bit adds), FP16 scan and head":
        lin_macs * E["int8_add"] + head_macs * fp16_mac + scan_fp,
    "  + shift-only decay (nearest / HARPQ: 1 shift)":
        lin_macs * E["int8_add"] + head_macs * fp16_mac + scan_fp - state_elems * E["fp16_mul"] + state_elems * shift_cost,
    "  + dyadic2 decay (2 shifts + 1 add)":
        lin_macs * E["int8_add"] + head_macs * fp16_mac + scan_fp - state_elems * E["fp16_mul"] + state_elems * (2 * shift_cost + E["int32_add"]),
}
base = cases["FP16 baseline (everything FP16)"]
ER = pd.DataFrame([dict(case=k, uJ_per_token=round(uJ(v), 1), vs_baseline=f"{v / base:.0%}") for k, v in cases.items()])
ER.to_csv(OUT / "energy_estimate.csv", index=False)
decay_fp = state_elems * E["fp16_mul"]
print(f"state elements per token: {state_elems:,} | linear MACs (ternary layers): {lin_macs:,} | head MACs: {head_macs:,}")
print(f"decay multiplies alone: {uJ(decay_fp):.2f} uJ/token = {decay_fp / base:.1%} of the FP16 baseline arithmetic")
print(ER.to_string(index=False))
print("\nReading: shift-only decay removes a few percent of arithmetic at most. After the linear layers are ternary, the FP16 output head is the largest arithmetic")
print("term left, so quantizing the head matters more for arithmetic energy than the decay does. Treat any decay-energy claim as small until measured.")


## 10. Budget planner and the commands for the missing runs

Use the measured throughput of your run log when it exists. The point of the planner is to choose a design you can finish: a matched ablation needs many runs, so the cheap protocol is usually the right place for the matrix and the long run is the headline demonstration.

In [ ]:
# ===== 10. Plan =====
meas = RUNS["tok_s_median"].dropna() if "tok_s_median" in RUNS else pd.Series(dtype=float)
by_proto = RUNS.dropna(subset=["tok_s_median"]).groupby("protocol").tok_s_median.median().to_dict() if len(meas) else {}
print("measured tokens/s by protocol:", {k: round(v) for k, v in by_proto.items()} or "none found")
tps = by_proto.get("run20M", DEFAULT_TOK_PER_S)
print(f"throughput used for the plan: {tps:,.0f} tokens/s", "" if "run20M" in by_proto else "(default; no 20M log found)")

have = set(zip(RUNS[RUNS.protocol == "run20M"].condition, RUNS[RUNS.protocol == "run20M"].seed))
todo = [(c, s) for c in PLAN["conditions"] for s in PLAN["seeds"] if (c, s) not in have]
hours_each = PLAN["tokens"] / tps / 3600 + 0.3          # + eval allowance
print(f"\nmatched matrix at {PLAN['tokens']:,} tokens: {len(todo)} runs still missing x {hours_each:.1f} h = {len(todo) * hours_each:.0f} h")
rows_ = []
for name, tok in [("fast 2M", 2_000_000), ("10M", 10_000_000), ("20M", PLAN["tokens"])]:
    for nseeds in (3, 5):
        n_runs = len(PLAN["conditions"]) * nseeds
        rows_.append(dict(protocol=name, seeds=nseeds, runs=n_runs, hours=round(n_runs * (tok / tps / 3600 + 0.3), 1)))
print(pd.DataFrame(rows_).to_string(index=False))
print("\nSuggested design: matrix (4 conditions x 5 seeds) at the cheap protocol for the ablation claims; one 20M run per best condition as the headline.")

def train_cmd(cond, seed, tokens, name_prefix="abl"):
    a = RUN20M_ARGS; name = f"{name_prefix}_{cond}_{tokens // 1_000_000}m_s{seed}"
    cmd = ["python", "-u", "train_qat.py", "--name", name, "--out", ROOTS["kaggle"], "--pretrained", "state-spaces/mamba2-130m"] + VARIANT_FLAGS[cond] + [
        "--ternary", "1", "--group-size", str(a["group_size"]), "--activation-aware", "1", "--state-weight", str(a["state_weight"]),
        "--state-loss-type", "hybrid", "--tokens", str(tokens), "--seq-len", str(a["seq_len"]), "--micro-bs", str(a["micro_bs"]),
        "--accum", str(a["accum"]), "--lr", str(a["lr"]), "--warmup", str(a["warmup"]), "--min-lr-frac", str(a["min_lr_frac"]),
        "--dtype", a["dtype"], "--val-seqs", str(a["val_seqs"]), "--eval-every", str(a["eval_every"]), "--save-every", str(a["save_every"]),
        "--seed", str(seed)]
    ev = ["python", "eval_lm.py", "--ckpt", f"{ROOTS['kaggle']}/{name}/last.pt", "--dataset", "HuggingFaceFW/fineweb-edu", "--subset", "sample-10BT",
          "--val-modulo", "100", "--ppl-length", "--dtype", a["dtype"], "--out", f"{ROOTS['kaggle']}/eval_{name}.json"]
    return " ".join(cmd), " ".join(ev)

print("\nCommands for the missing runs (same settings as your 20M job; run the eval with the SAME dtype for every condition and for the teacher):")
for c, s in todo[:6]:
    t, e = train_cmd(c, s, PLAN["tokens"]); print(f"\n# {c} seed {s}\n{t}\n{e}")
if len(todo) > 6: print(f"\n... {len(todo) - 6} more with the same pattern")
if "fp_decay" in PLAN["conditions"]:
    print("\nNote: fp_decay uses --decay exp. Confirm that value in `python train_qat.py --help` before starting a long run.")


In [ ]:
# ===== 11. Report =====
lines = ["# Shift-Scan results analysis", f"Generated {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())}.", ""]
lines += ["## Data inventory", md_table(inv), ""]
if notes_skipped: lines += ["Skipped (files not found yet):"] + [f"- {s}" for s in notes_skipped] + [""]
lines += ["## Problems and caveats"]
caveats = list(problems)
if RUNS.groupby("protocol").seed.nunique().max() < MIN_SEEDS_FOR_VERDICT:
    caveats.append(f"no protocol has {MIN_SEEDS_FOR_VERDICT} seeds per condition yet; paired verdicts are withheld")
if RUNS.partial.any(): caveats.append(f"{int(RUNS.partial.sum())} row(s) are partial (typed or copied values) and are used only for the metrics they contain")
if not CH.empty and (CH.above_chance_2se == False).any():
    caveats.append("not clearly above chance (2 SE): " + ", ".join(sorted(set(CH[CH.above_chance_2se == False].task))))
caveats.append("different protocols (token budget, sequence length, eval dtype) are never pooled")
caveats.append("kernel speed and energy are not measured; the energy section is an analytic estimate")
lines += [f"- {c}" for c in caveats] + [""]
lines += ["## Per-condition results", md_table(MAIN), ""]
if not GAP.empty: lines += ["## Gap to teacher", md_table(GAP.round(4)), ""]
if not CH.empty: lines += ["## Multiple-choice scores against chance", md_table(CH.round(4)), ""]
if not PAIRED.empty:
    lines += ["## Paired ablations (primary metric and student PPL)",
              md_table(PAIRED[PAIRED.metric.isin([PRIMARY_METRIC, "student_ppl"])][["protocol", "A", "B", "metric", "n", "mean_adv", "rel_pct", "a_better", "p", "p_min", "holm_p", "verdict"]].round(4)), ""]
if not CAL.empty: lines += ["## Decay calibration", md_table(CAL.round(5)), ""]
lines += ["## Compression scenarios (size arithmetic only)", md_table(SC), "", "## Arithmetic energy estimate (analytic)", md_table(ER), ""]

claims = {"supported": [], "not_supported": []}
if not PAIRED.empty:
    for _, r in PAIRED[PAIRED.metric == PRIMARY_METRIC].iterrows():
        (claims["supported"] if ("better than" in r.verdict) else claims["not_supported"]).append(f"[{r.protocol}] {r.A} vs {r.B}: {r.verdict}")
claims["not_supported"] += ["any speed claim (no end-to-end hardware measurement)", "any energy claim (analytic only)"]
lines += ["## What the data supports right now", "Supported:"] + [f"- {c}" for c in claims["supported"]] + (["- (nothing yet at the declared threshold)"] if not claims["supported"] else [])
lines += ["", "Not supported:"] + [f"- {c}" for c in claims["not_supported"]]
(OUT / "analysis_report.md").write_text("\n".join(lines), encoding="utf-8")

summary = dict(generated=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), n_runs=int(len(RUNS)), n_partial=int(RUNS.partial.sum()),
               protocols=sorted(RUNS.protocol.unique()), problems=problems, caveats=caveats, claims=claims,
               config=dict(primary_metric=PRIMARY_METRIC, ablations=ABLATIONS, min_seeds=MIN_SEEDS_FOR_VERDICT, alpha=ALPHA))
(OUT / "analysis_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("\n".join(lines[:12]), "\n...")
print("\nwrote:", ", ".join(sorted(p.name for p in OUT.glob("*") if p.is_file())), "+ figs/")
bundle = shutil.make_archive("analysis_bundle", "zip", OUT); print("bundle:", bundle)
if IN_COLAB: files.download(bundle)
